# Bibliotecas
 - Coleta de dados validados como boatos = falso
 - Usa a Google Fact Check Tools API para buscar afirmações já verificadas
 - Suporta MODO_TESTE para validação rápida antes da coleta completa

In [1]:
import os
import time
import random
import requests
import pandas as pd

from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

# Google Fact Check

## Configuração e modo de execução

**MODO_TESTE = True** → roda apenas os primeiros `N_TERMOS_TESTE` termos com no máximo
`MAX_PAGINAS_TESTE` páginas cada. Use para validar que a API está respondendo antes
de executar a coleta completa.

**MODO_TESTE = False** → coleta completa com todos os termos e configurações de produção.

In [2]:
# ==============================================================
# MODO DE EXECUÇÃO — altere aqui antes de rodar
# ==============================================================
MODO_TESTE        = False  # True = teste rápido | False = coleta completa
N_TERMOS_TESTE    = 5     # quantos termos usar no modo teste
MAX_PAGINAS_TESTE = 2     # máximo de páginas por termo no modo teste

# ==============================================================
# CHAVE DA API
# ==============================================================
load_dotenv("../google-factcheck-api-key.env")

API_KEY = os.getenv("GOOGLE_FACTCHECK_API_KEY")

if not API_KEY:
    raise ValueError("Chave da API não encontrada. Verifique o arquivo google-factcheck-api-key.env")

# ==============================================================
# PARÂMETROS DA API — produção
# ==============================================================
URL          = "https://factchecktools.googleapis.com/v1alpha1/claims:search"
PAGE_SIZE    = 50    # máximo permitido pela API
MAX_AGE_DAYS = 1825  # 5 anos de histórico
MAX_PAGINAS  = 10    # máximo de páginas por termo em produção

# ==============================================================
# PARÂMETROS DE RETRY — proteção contra erro 503
# ==============================================================
MAX_TENTATIVAS        = 5  # tentativas por requisição antes de desistir
BACKOFF_BASE_SEGUNDOS = 5  # espera base em segundos (dobra a cada tentativa)
#
# Sequência de espera com jitter ±1s:
#   tentativa 1 →  5s ± 1s  (~4–6s)
#   tentativa 2 → 10s ± 1s  (~9–11s)
#   tentativa 3 → 20s ± 1s  (~19–21s)
#   tentativa 4 → 40s ± 1s  (~39–41s)
#   tentativa 5 → 80s ± 1s  (~79–81s)

# ==============================================================
# INTERVALOS ENTRE REQUISIÇÕES — reduz throttling
# ==============================================================
SLEEP_ENTRE_PAGINAS = 2.0  # segundos entre páginas do mesmo termo
SLEEP_ENTRE_TERMOS  = 5.0  # segundos entre termos diferentes

# Aplica limites do modo teste se ativado
max_paginas_por_termo = MAX_PAGINAS_TESTE if MODO_TESTE else MAX_PAGINAS

print(f"Modo de execução : {'TESTE' if MODO_TESTE else 'PRODUÇÃO'}")
print(f"Max páginas/termo: {max_paginas_por_termo}")
print(f"Max age days     : {MAX_AGE_DAYS}")
print(f"Max tentativas   : {MAX_TENTATIVAS}  (backoff: "
      + ", ".join(f"{BACKOFF_BASE_SEGUNDOS * 2**i}s" for i in range(MAX_TENTATIVAS)) + ")")
print(f"Sleep pág/termo  : {SLEEP_ENTRE_PAGINAS}s / {SLEEP_ENTRE_TERMOS}s")

Modo de execução : PRODUÇÃO
Max páginas/termo: 10
Max age days     : 1825
Max tentativas   : 5  (backoff: 5s, 10s, 20s, 40s, 80s)
Sleep pág/termo  : 2.0s / 5.0s


## Termos de busca

Organizados por categoria para facilitar manutenção e expansão futura.
Cada categoria pode ser desabilitada individualmente antes de uma coleta parcial.

In [3]:
# ==============================================================
# QUERIES_SUGERIDAS — 83 queries estruturadas por tema e subtema
# (substitui as listas categóricas da versão anterior)
# ==============================================================
QUERIES_SUGERIDAS = [
    # ─── Pautas trabalhistas ──────────────────────────────────────────────
    {"tema": "trabalhista",         "subtema": "jornada_trabalho",   "query": "escala 6x1"},
    {"tema": "trabalhista",         "subtema": "jornada_trabalho",   "query": "fim da escala 6x1"},
    {"tema": "trabalhista",         "subtema": "jornada_trabalho",   "query": "jornada de trabalho"},
    {"tema": "trabalhista",         "subtema": "legislacao",         "query": "CLT"},
    {"tema": "trabalhista",         "subtema": "remuneracao",        "query": "salário mínimo"},
    {"tema": "trabalhista",         "subtema": "previdencia",        "query": "INSS"},
    {"tema": "trabalhista",         "subtema": "previdencia",        "query": "aposentadoria"},
    {"tema": "trabalhista",         "subtema": "beneficios",         "query": "FGTS"},
    {"tema": "trabalhista",         "subtema": "empreendedorismo",   "query": "MEI"},
    {"tema": "trabalhista",         "subtema": "legislacao",         "query": "pejotização"},
    # ─── Eleições e Justiça Eleitoral ────────────────────────────────────
    {"tema": "eleitoral",           "subtema": "eleicoes_2026",      "query": "eleições 2026"},
    {"tema": "eleitoral",           "subtema": "sistema_voto",       "query": "urna eletrônica"},
    {"tema": "eleitoral",           "subtema": "fraude_eleitoral",   "query": "fraude eleitoral"},
    {"tema": "eleitoral",           "subtema": "justica_eleitoral",  "query": "TSE"},
    {"tema": "eleitoral",           "subtema": "sistema_voto",       "query": "voto impresso"},
    {"tema": "eleitoral",           "subtema": "cadastro_eleitoral", "query": "biometria eleitoral"},
    {"tema": "eleitoral",           "subtema": "campanha",           "query": "propaganda eleitoral"},
    {"tema": "eleitoral",           "subtema": "pesquisas",          "query": "pesquisa eleitoral"},
    {"tema": "eleitoral",           "subtema": "desinformacao",      "query": "fake news eleitoral"},
    {"tema": "eleitoral",           "subtema": "desinformacao",      "query": "deepfake eleições"},
    {"tema": "eleitoral",           "subtema": "desinformacao",      "query": "inteligência artificial eleições"},
    # ─── Congresso, governo e instituições ───────────────────────────────
    {"tema": "institucional",       "subtema": "poder_judiciario",   "query": "STF"},
    {"tema": "institucional",       "subtema": "poder_judiciario",   "query": "Alexandre de Moraes"},
    {"tema": "institucional",       "subtema": "poder_judiciario",   "query": "Supremo Tribunal Federal"},
    {"tema": "institucional",       "subtema": "politica_criminal",  "query": "8 de janeiro"},
    {"tema": "institucional",       "subtema": "policia",            "query": "Polícia Federal"},
    {"tema": "institucional",       "subtema": "poder_executivo",    "query": "impeachment"},
    {"tema": "institucional",       "subtema": "legislativo",        "query": "CPI"},
    {"tema": "institucional",       "subtema": "legislativo",        "query": "PEC"},
    {"tema": "institucional",       "subtema": "legislativo",        "query": "Congresso Nacional"},
    {"tema": "institucional",       "subtema": "legislativo",        "query": "Câmara dos Deputados"},
    {"tema": "institucional",       "subtema": "legislativo",        "query": "Senado Federal"},
    {"tema": "institucional",       "subtema": "orcamento",          "query": "emendas parlamentares"},
    {"tema": "institucional",       "subtema": "orcamento",          "query": "orçamento secreto"},
    # ─── Economia popular ─────────────────────────────────────────────────
    {"tema": "economia_popular",    "subtema": "pagamentos_digitais","query": "Pix"},
    {"tema": "economia_popular",    "subtema": "politica_monetaria", "query": "Banco Central"},
    {"tema": "economia_popular",    "subtema": "indices",            "query": "inflação"},
    {"tema": "economia_popular",    "subtema": "politica_monetaria", "query": "taxa Selic"},
    {"tema": "economia_popular",    "subtema": "combustiveis",       "query": "preço da gasolina"},
    {"tema": "economia_popular",    "subtema": "assistencia_social", "query": "Bolsa Família"},
    {"tema": "economia_popular",    "subtema": "assistencia_social", "query": "Auxílio Brasil"},
    {"tema": "economia_popular",    "subtema": "habitacao",          "query": "Minha Casa Minha Vida"},
    {"tema": "economia_popular",    "subtema": "tributacao",         "query": "reforma tributária"},
    {"tema": "economia_popular",    "subtema": "tributacao",         "query": "imposto de renda"},
    {"tema": "economia_popular",    "subtema": "comercio_exterior",  "query": "taxação da Shein"},
    {"tema": "economia_popular",    "subtema": "regulacao",          "query": "taxação de bets"},
    # ─── Saúde, educação e segurança pública ──────────────────────────────
    {"tema": "social",              "subtema": "saude",              "query": "SUS"},
    {"tema": "social",              "subtema": "saude",              "query": "vacina"},
    {"tema": "social",              "subtema": "saude",              "query": "dengue"},
    {"tema": "social",              "subtema": "saude",              "query": "Covid"},
    {"tema": "social",              "subtema": "educacao",           "query": "Enem"},
    {"tema": "social",              "subtema": "educacao",           "query": "Fies"},
    {"tema": "social",              "subtema": "educacao",           "query": "Prouni"},
    {"tema": "social",              "subtema": "seguranca",          "query": "segurança pública"},
    {"tema": "social",              "subtema": "seguranca",          "query": "saidinha temporária"},
    {"tema": "social",              "subtema": "seguranca",          "query": "porte de armas"},
    # ─── Figuras políticas ────────────────────────────────────────────────
    {"tema": "politica_figuras",    "subtema": "governo_federal",    "query": "Lula"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Bolsonaro"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Jair Bolsonaro"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Michelle Bolsonaro"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Eduardo Bolsonaro"},
    {"tema": "politica_figuras",    "subtema": "governos_estaduais", "query": "Tarcísio de Freitas"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Guilherme Boulos"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Pablo Marçal"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Nikolas Ferreira"},
    {"tema": "politica_figuras",    "subtema": "governo_federal",    "query": "André Janones"},
    {"tema": "politica_figuras",    "subtema": "legislativo",        "query": "Arthur Lira"},
    {"tema": "politica_figuras",    "subtema": "legislativo",        "query": "Rodrigo Pacheco"},
    {"tema": "politica_figuras",    "subtema": "oposicao",           "query": "Sergio Moro"},
    # ─── Movimentos, partidos e grupos políticos ──────────────────────────
    {"tema": "politica_partidos",   "subtema": "movimentos",         "query": "MBL"},
    {"tema": "politica_partidos",   "subtema": "movimentos",         "query": "Movimento Brasil Livre"},
    {"tema": "politica_partidos",   "subtema": "partidos",           "query": "PT"},
    {"tema": "politica_partidos",   "subtema": "partidos",           "query": "PL"},
    {"tema": "politica_partidos",   "subtema": "partidos",           "query": "PSOL"},
    {"tema": "politica_partidos",   "subtema": "partidos",           "query": "MDB"},
    {"tema": "politica_partidos",   "subtema": "partidos",           "query": "Novo"},
    {"tema": "politica_partidos",   "subtema": "legislativo",        "query": "Centrão"},
    {"tema": "politica_partidos",   "subtema": "ideologia",          "query": "bolsonarismo"},
    {"tema": "politica_partidos",   "subtema": "ideologia",          "query": "lulismo"},
    {"tema": "politica_partidos",   "subtema": "investigacoes",      "query": "lava jato"},
    # ─── Temas virais e narrativas de redes sociais ───────────────────────
    {"tema": "desinformacao_viral", "subtema": "narrativas_whatsapp","query": "kit gay"},
    {"tema": "desinformacao_viral", "subtema": "narrativas_whatsapp","query": "ideologia de gênero"},
    {"tema": "desinformacao_viral", "subtema": "narrativas_whatsapp","query": "comunismo no Brasil"},
    {"tema": "desinformacao_viral", "subtema": "narrativas_whatsapp","query": "Foro de São Paulo"},
    {"tema": "desinformacao_viral", "subtema": "narrativas_whatsapp","query": "Venezuela"},
    {"tema": "desinformacao_viral", "subtema": "liberdade_expressao","query": "censura"},
    {"tema": "desinformacao_viral", "subtema": "liberdade_expressao","query": "liberdade de expressão"},
    {"tema": "desinformacao_viral", "subtema": "regulacao_digital",  "query": "bloqueio de redes sociais"},
    {"tema": "desinformacao_viral", "subtema": "conteudo_falso",     "query": "vídeo manipulado"},
    {"tema": "desinformacao_viral", "subtema": "conteudo_falso",     "query": "áudio falso"},
    {"tema": "desinformacao_viral", "subtema": "conteudo_falso",     "query": "print falso"},
    {"tema": "desinformacao_viral", "subtema": "conteudo_falso",     "query": "montagem política"},
]

# Lookup rápido: query (lower) → {tema, subtema}
_QUERY_META = {q["query"].lower(): q for q in QUERIES_SUGERIDAS}

# Lista plana para manter compatibilidade com o loop de coleta
termos_busca = [q["query"] for q in QUERIES_SUGERIDAS]

# Contagens por tema
_temas = {}
for q in QUERIES_SUGERIDAS:
    _temas[q["tema"]] = _temas.get(q["tema"], 0) + 1

print(f"Total de queries configuradas: {len(termos_busca)}")
for tema, cnt in sorted(_temas.items()):
    print(f"  {tema:<24} : {cnt}")


Total de queries configuradas: 92
  desinformacao_viral      : 12
  economia_popular         : 12
  eleitoral                : 11
  institucional            : 13
  politica_figuras         : 13
  politica_partidos        : 11
  social                   : 10
  trabalhista              : 10


## Seleção de termos conforme modo de execução

In [4]:
if MODO_TESTE:
    # No modo teste, usa apenas os primeiros N termos da lista unificada
    # para validar que a API está respondendo e o pipeline funciona
    termos_para_coletar = termos_busca[:N_TERMOS_TESTE]
    print(f"MODO TESTE — coletando {len(termos_para_coletar)} termos:")
else:
    termos_para_coletar = termos_busca
    print(f"MODO PRODUÇÃO — coletando todos os {len(termos_para_coletar)} termos:")

for t in termos_para_coletar:
    print(f"  - {t}")

MODO PRODUÇÃO — coletando todos os 92 termos:
  - escala 6x1
  - fim da escala 6x1
  - jornada de trabalho
  - CLT
  - salário mínimo
  - INSS
  - aposentadoria
  - FGTS
  - MEI
  - pejotização
  - eleições 2026
  - urna eletrônica
  - fraude eleitoral
  - TSE
  - voto impresso
  - biometria eleitoral
  - propaganda eleitoral
  - pesquisa eleitoral
  - fake news eleitoral
  - deepfake eleições
  - inteligência artificial eleições
  - STF
  - Alexandre de Moraes
  - Supremo Tribunal Federal
  - 8 de janeiro
  - Polícia Federal
  - impeachment
  - CPI
  - PEC
  - Congresso Nacional
  - Câmara dos Deputados
  - Senado Federal
  - emendas parlamentares
  - orçamento secreto
  - Pix
  - Banco Central
  - inflação
  - taxa Selic
  - preço da gasolina
  - Bolsa Família
  - Auxílio Brasil
  - Minha Casa Minha Vida
  - reforma tributária
  - imposto de renda
  - taxação da Shein
  - taxação de bets
  - SUS
  - vacina
  - dengue
  - Covid
  - Enem
  - Fies
  - Prouni
  - segurança pública
  - sa

## Função de requisição com retry e backoff exponencial

Em caso de erro 503/504/500/429, a função aguarda um tempo crescente antes de
tentar novamente — até `MAX_TENTATIVAS` vezes. Isso evita que termos inteiros
sejam perdidos silenciosamente por instabilidades temporárias da API.

**Sequência de espera** (backoff `5 × 2^(n-1)` + jitter ±1s):

| Tentativa | Espera base | Com jitter |
|-----------|-------------|------------|
| 1 | 5s | ~4–6s |
| 2 | 10s | ~9–11s |
| 3 | 20s | ~19–21s |
| 4 | 40s | ~39–41s |
| 5 | 80s | ~79–81s |

Erros não-recuperáveis (400, 401, 403) abortam imediatamente sem retry.

In [5]:
def requisitar_com_retry(url: str, params: dict) -> requests.Response | None:
    """
    Faz uma requisição GET com até MAX_TENTATIVAS tentativas em erros recuperáveis.

    Backoff: espera = BACKOFF_BASE_SEGUNDOS * 2^(tentativa-1) + jitter ±1s
      tentativa 1 →  5s ± 1s
      tentativa 2 → 10s ± 1s
      tentativa 3 → 20s ± 1s
      tentativa 4 → 40s ± 1s
      tentativa 5 → 80s ± 1s

    Retorna o Response em caso de sucesso (HTTP 200).
    Retorna None se todas as tentativas falharem por erro recuperável.
    Retorna o Response imediatamente em erros não-recuperáveis (400, 401, 403).
    """
    ERROS_RECUPERAVEIS = {429, 500, 503, 504}

    for tentativa in range(1, MAX_TENTATIVAS + 1):

        resposta = None
        try:
            resposta = requests.get(url, params=params, timeout=30)
        except requests.exceptions.RequestException as exc:
            print(f"    [tentativa {tentativa}/{MAX_TENTATIVAS}] Erro de conexão: {exc}")

        # Sucesso
        if resposta is not None and resposta.status_code == 200:
            return resposta

        status = resposta.status_code if resposta is not None else "conexão falhou"

        # Erro não-recuperável — não adianta tentar de novo
        if resposta is not None and resposta.status_code not in ERROS_RECUPERAVEIS:
            print(f"    Erro não-recuperável: HTTP {status} (sem retry)")
            return resposta

        # Erro recuperável — calcula espera e tenta de novo (se ainda há tentativas)
        if tentativa < MAX_TENTATIVAS:
            espera_base = BACKOFF_BASE_SEGUNDOS * (2 ** (tentativa - 1))
            jitter      = random.uniform(-1.0, 1.0)
            espera_total = round(max(1.0, espera_base + jitter), 1)
            print(
                f"    [tentativa {tentativa}/{MAX_TENTATIVAS}] HTTP {status} "
                f"— aguardando {espera_total}s antes de tentar novamente..."
            )
            time.sleep(espera_total)
        else:
            print(
                f"    [tentativa {tentativa}/{MAX_TENTATIVAS}] HTTP {status} "
                f"— todas as tentativas esgotadas."
            )

    # Todas as tentativas falharam
    return None


print("Função de retry definida (5 tentativas, backoff 5s→10s→20s→40s→80s ± 1s).")

Função de retry definida (5 tentativas, backoff 5s→10s→20s→40s→80s ± 1s).


## Coleta — API Google Fact Check

In [6]:
registros            = []
termos_com_resultado = []   # termos que retornaram ao menos 1 claim
termos_sem_resultado = []   # termos que retornaram 0 claims sem erro
termos_com_erro      = []   # termos que falharam mesmo após todos os retries
detalhes_erros       = []   # log detalhado de cada falha

total_termos = len(termos_para_coletar)

for idx, termo in enumerate(termos_para_coletar, start=1):

    print(f"\n[{idx}/{total_termos}] Consultando: {termo}")

    page_token   = None
    pagina_atual = 1
    claims_termo = 0
    erro_neste_termo = False

    while pagina_atual <= max_paginas_por_termo:

        params = {
            "query"       : termo,
            "languageCode": "pt",
            "pageSize"    : PAGE_SIZE,
            "maxAgeDays"  : MAX_AGE_DAYS,
            "key"         : API_KEY,
        }

        if page_token:
            params["pageToken"] = page_token

        resposta = requisitar_com_retry(URL, params)

        # Falha persistente após todos os retries
        if resposta is None or resposta.status_code != 200:
            status_code = resposta.status_code if resposta is not None else "conexão falhou"
            mensagem    = ""
            if resposta is not None:
                try:
                    mensagem = resposta.json().get("error", {}).get("message", "")
                except Exception:
                    mensagem = resposta.text[:200]

            print(f"  ✗ Abortando '{termo}' (HTTP {status_code}, pág. {pagina_atual})")

            termos_com_erro.append(termo)
            detalhes_erros.append({
                "termo"      : termo,
                "pagina"     : pagina_atual,
                "status_code": status_code,
                "mensagem"   : mensagem,
                "horario"    : datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
            })
            erro_neste_termo = True
            break

        dados  = resposta.json()
        claims = dados.get("claims", [])

        print(f"  Página {pagina_atual} — {len(claims)} claims")
        claims_termo += len(claims)

        for claim in claims:
            texto_claim = claim.get("text", "")
            data_claim  = claim.get("claimDate", "")

            for review in claim.get("claimReview", []):
                _meta = _QUERY_META.get(termo.lower(), {})
                registros.append({
                    "termo_busca"       : termo,
                    "query_matched"     : _meta.get("query", termo),
                    "tema_query"        : _meta.get("tema", ""),
                    "subtema_query"     : _meta.get("subtema", ""),
                    "texto_afirmacao"   : texto_claim,
                    "data_claim"        : data_claim,
                    "fonte_verificacao" : review.get("publisher", {}).get("name", ""),
                    "url_checagem"      : review.get("url", ""),
                    "avaliacao_original": review.get("textualRating", ""),
                    "data_publicacao"   : review.get("reviewDate", ""),
                    "endpoint_consulta" : "claims:search",
                    "query_api"         : termo,
                    "languageCode"      : "pt",
                    "pageSize"          : PAGE_SIZE,
                    "maxAgeDays"        : MAX_AGE_DAYS,
                    "data_coleta"       : datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
                    "origem_pipeline"   : "GOOGLE_FACTCHECK",
                })

        page_token = dados.get("nextPageToken")
        if not page_token:
            break

        pagina_atual += 1
        time.sleep(SLEEP_ENTRE_PAGINAS)

    # Classifica o termo conforme resultado
    if not erro_neste_termo:
        if claims_termo > 0:
            termos_com_resultado.append(termo)
        else:
            termos_sem_resultado.append(termo)
            print(f"  — Nenhuma claim retornada para '{termo}'")

    # Pausa entre termos (somente se não for o último)
    if idx < total_termos:
        time.sleep(SLEEP_ENTRE_TERMOS)

# ==============================================================
# RESUMO FINAL DA COLETA
# ==============================================================
separador = "=" * 56
print(f"\n{separador}")
print(f"  RESUMO DA COLETA — {'TESTE' if MODO_TESTE else 'PRODUÇÃO'}")
print(separador)
print(f"  Total de registros brutos coletados : {len(registros)}")
print(f"  Termos com resultado (≥1 claim)     : {len(termos_com_resultado)}")
print(f"  Termos sem resultado (0 claims)     : {len(termos_sem_resultado)}")
print(f"  Termos com erro (falha persistente) : {len(termos_com_erro)}")
print(separador)

if termos_sem_resultado:
    print(f"\n  Termos sem resultado:")
    for t in termos_sem_resultado:
        print(f"    - {t}")

if termos_com_erro:
    print(f"\n  Termos com erro — rode novamente ou ajuste a lista:")
    for d in detalhes_erros:
        print(f"    - {d['termo']:35s}  HTTP {d['status_code']}  pág.{d['pagina']}  {d['horario']}")

print(f"\n{separador}")


[1/92] Consultando: escala 6x1
  Página 1 — 7 claims

[2/92] Consultando: fim da escala 6x1
  Página 1 — 8 claims

[3/92] Consultando: jornada de trabalho
  Página 1 — 3 claims

[4/92] Consultando: CLT
  Página 1 — 2 claims

[5/92] Consultando: salário mínimo
  Página 1 — 44 claims

[6/92] Consultando: INSS
  Página 1 — 50 claims
  Página 2 — 39 claims

[7/92] Consultando: aposentadoria
  Página 1 — 50 claims
  Página 2 — 35 claims

[8/92] Consultando: FGTS
  Página 1 — 23 claims

[9/92] Consultando: MEI
  Página 1 — 11 claims

[10/92] Consultando: pejotização
  Página 1 — 0 claims
  — Nenhuma claim retornada para 'pejotização'

[11/92] Consultando: eleições 2026
  Página 1 — 21 claims

[12/92] Consultando: urna eletrônica
  Página 1 — 50 claims
  Página 2 — 8 claims

[13/92] Consultando: fraude eleitoral
  Página 1 — 50 claims
  Página 2 — 50 claims

[14/92] Consultando: TSE
  Página 1 — 50 claims
  Página 2 — 50 claims
  Página 3 — 0 claims

[15/92] Consultando: voto impresso
  Pági

## DataFrame raw e deduplicação interna

Deduplica dentro desta coleta antes de salvar.
A deduplicação entre coletas diferentes é responsabilidade do notebook de curadoria.

In [7]:
df_google_raw = pd.DataFrame(registros)

print(f"Registros brutos coletados nesta execução: {len(df_google_raw)}")

if len(df_google_raw) > 0:
    # Deduplicação dentro desta coleta:
    # mesma afirmação verificada pela mesma fonte = registro duplicado
    antes = len(df_google_raw)
    df_google_raw = df_google_raw.drop_duplicates(
        subset=["texto_afirmacao", "fonte_verificacao"],
        keep="first"
    ).reset_index(drop=True)
    print(f"Duplicatas removidas nesta coleta  : {antes - len(df_google_raw)}")
    print(f"Registros únicos para salvar       : {len(df_google_raw)}")

    print(f"\nDistribuição por avaliacao_original (top 10):")
    print(df_google_raw["avaliacao_original"].value_counts().head(10))

    print(f"\nDistribuição por fonte_verificacao (top 10):")
    print(df_google_raw["fonte_verificacao"].value_counts().head(10))

    display(df_google_raw.head(3))
else:
    print("Nenhum registro coletado. Verifique a chave da API e os termos de busca.")

Registros brutos coletados nesta execução: 6433
Duplicatas removidas nesta coleta  : 2341
Registros únicos para salvar       : 4092

Distribuição por avaliacao_original (top 10):
avaliacao_original
Falso               1415
falso               1098
Enganoso             674
Errado               201
não_é_bem_assim      119
Distorcido            92
Fora de contexto      55
Insustentável         49
Enganador             34
Sem contexto          24
Name: count, dtype: int64

Distribuição por fonte_verificacao (top 10):
fonte_verificacao
Aos Fatos           1262
Estadão              860
UOL Notícias         706
AFP Checamos         587
Projeto Comprova     277
Observador           237
Folha - UOL           78
BOL - UOL             53
Metrópoles            13
Agência Tatu           9
Name: count, dtype: int64


,termo_busca,query_matched,tema_query,subtema_query,texto_afirmacao,data_claim,fonte_verificacao,url_checagem,avaliacao_original,data_publicacao,url_consulta,data_coleta,origem_pipeline
0,escala 6x1,escala 6x1,trabalhista,jornada_trabalho,Flávio Bolsonaro disse que vai revogar o fim d...,2026-04-24T00:00:00Z,Aos Fatos,https://www.aosfatos.org/noticias/flavio-bolso...,falso,2026-04-24T00:00:00Z,https://factchecktools.googleapis.com/v1alpha1...,30/05/2026 21:30:19,GOOGLE_FACTCHECK
1,escala 6x1,escala 6x1,trabalhista,jornada_trabalho,Vídeo mostra comemoração em São Paulo pela apr...,2026-05-29T00:00:00Z,Aos Fatos,https://www.aosfatos.org/noticias/comemoracao-...,falso,2026-05-29T00:00:00Z,https://factchecktools.googleapis.com/v1alpha1...,30/05/2026 21:30:19,GOOGLE_FACTCHECK
2,escala 6x1,escala 6x1,trabalhista,jornada_trabalho,PEC do fim da escala 6x1 não tem estudo de via...,2024-11-14T18:59:00Z,Estadão,https://www.estadao.com.br/estadao-verifica/pe...,Enganoso,,https://factchecktools.googleapis.com/v1alpha1...,30/05/2026 21:30:19,GOOGLE_FACTCHECK


## Extração — salvar CSV raw com timestamp

In [8]:
nome_pipeline = "pipeline_falso_google_factcheck"
nome_base     = "google_factcheck"
sufixo_modo   = "_TESTE" if MODO_TESTE else ""
data_agora    = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

pasta_raw = Path(f"../dados/{nome_pipeline}/raw")
pasta_raw.mkdir(parents=True, exist_ok=True)

# ==============================================================
# 1. CSV PRINCIPAL — registros coletados
# ==============================================================
if len(df_google_raw) == 0:
    print("Nenhum registro para salvar. Abortando exportação do CSV principal.")
else:
    caminho_principal = pasta_raw / f"{nome_base}_raw{sufixo_modo}_{data_agora}.csv"
    df_google_raw.to_csv(caminho_principal, index=False, encoding="utf-8-sig")

    print(f"CSV principal salvo : {caminho_principal}")
    print(f"Registros exportados: {len(df_google_raw)}")
    print(f"Modo                : {'TESTE' if MODO_TESTE else 'PRODUÇÃO'}")
    print(f"Data/hora           : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

# ==============================================================
# 2. CSV DE LOG DE ERROS — apenas se houve falhas
# ==============================================================
if detalhes_erros:
    df_erros = pd.DataFrame(detalhes_erros)
    caminho_erros = pasta_raw / f"{nome_base}_erros{sufixo_modo}_{data_agora}.csv"
    df_erros.to_csv(caminho_erros, index=False, encoding="utf-8-sig")

    print(f"\nCSV de erros salvo  : {caminho_erros}")
    print(f"Termos com erro     : {len(df_erros)}")
    print("\nConteúdo do log de erros:")
    print(df_erros[["termo", "pagina", "status_code", "horario"]].to_string(index=False))
else:
    print("\nNenhum erro registrado — CSV de log de erros não gerado.")

# ==============================================================
# 3. INSTRUÇÃO FINAL
# ==============================================================
if MODO_TESTE:
    print(f"\n{'=' * 56}")
    print(f"  MODO TESTE concluído.")
    print(f"  Para rodar a coleta completa:")
    print(f"    1. Defina MODO_TESTE = False na célula de configuração")
    print(f"    2. Execute o notebook do início")
    if detalhes_erros:
        print(f"\n  Termos com erro para tentar de novo:")
        print(f"    Copie a lista abaixo e use em uma coleta de retry:")
        termos_erro_str = "\n".join(f'    "{t}",' for t in termos_com_erro)
        print(termos_erro_str)
    print(f"{'=' * 56}")

CSV principal salvo : ..\dados\pipeline_falso_google_factcheck\raw\google_factcheck_raw_2026-05-30_21-44-39.csv
Registros exportados: 4092
Modo                : PRODUÇÃO
Data/hora           : 30/05/2026 21:44:39

Nenhum erro registrado — CSV de log de erros não gerado.


## Validação de Segurança dos Outputs

Verifica que nenhum arquivo gerado contém padrões sensíveis (API key, `key=`, `AIza...`).
Deve imprimir **'Nenhuma API key encontrada'** para que o arquivo seja seguro para commit.

In [ ]:
import re as _re

_PADROES_SENSIVEIS = [
    r"key=[A-Za-z0-9_\-]{10,}",
    r"AIza[A-Za-z0-9_\-]{10,}",
    r"GOOGLE_API_KEY",
    r"FACTCHECK_API_KEY",
]

def _verificar_arquivo(caminho):
    """Retorna lista de achados sensíveis (linha, padrão) — vazia = seguro."""
    achados = []
    try:
        with open(caminho, encoding="utf-8-sig", errors="replace") as _f:
            for n_linha, linha in enumerate(_f, 1):
                for pat in _PADROES_SENSIVEIS:
                    if _re.search(pat, linha):
                        achados.append((n_linha, pat, linha[:120].strip()))
                        break
    except Exception as exc:
        achados.append((0, "ERRO_LEITURA", str(exc)))
    return achados

_arquivos_verificar = []
if len(df_google_raw) > 0:
    _arquivos_verificar.append(caminho_principal)
if detalhes_erros:
    _arquivos_verificar.append(caminho_erros)

_total_achados = 0
print("=== Validação de Segurança dos Outputs ===")
print(f"Padrões verificados: {_PADROES_SENSIVEIS}")
print(f"Coluna url_consulta removida : SIM (substituída por endpoint_consulta + query_api)")
print()
for _arq in _arquivos_verificar:
    _achados = _verificar_arquivo(_arq)
    _total_achados += len(_achados)
    if _achados:
        print(f"  ALERTA em {_arq.name}:")
        for _ln, _pat, _trecho in _achados[:5]:
            print(f"    linha {_ln}: [{_pat}] {_trecho[:80]}")
    else:
        print(f"  OK: {_arq.name} — nenhum padrão sensível encontrado")

if _total_achados == 0:
    print()
    print("Validação de segurança: nenhuma API key encontrada nos outputs.")
    print("Arquivos seguros para commit.")
else:
    print()
    print(f"ATENÇÃO: {_total_achados} ocorrência(s) de padrão sensível encontrada(s).")
    print("NÃO commitar estes arquivos antes de revisar e sanitizar.")
